# Data Processing


In [ ]:
# Standard library
import os
import random
import time
import zipfile
from collections import defaultdict

# Numerical and plotting
import numpy as np
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split, TensorDataset

# Torchvision
import torchvision
from torchvision import datasets, transforms, models

# Scikit-learn
import sklearn
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/Datasets/APS360_archive_1.zip"

In [ ]:
path = '/content/dataset'

# Transform data

transform_augment = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Numbers from ImageNet
])

# Load dataset
augment_dataset = torchvision.datasets.ImageFolder(root=path, transform=transform_augment)
dataset = torchvision.datasets.ImageFolder(root=path, transform=transform)

torch.manual_seed(42) # Ensure same seed for reproducible results

train_index, val_index, test_index = random_split(list(range(len(dataset))), [0.8, 0.1, 0.1]) # 80, 10, 10 split

train_set = Subset(augment_dataset, train_index)
val_set = Subset(dataset, val_index)
test_set = Subset(dataset, test_index)

print(f'Training Set Size: {len(train_set)}')
print(f'Validation Set Size: {len(val_set)}')
print(f'Test Set Size: {len(test_set)}')

img, label = train_set[0]
img = img.permute(1, 2, 0)
plt.title("First Training Sample")
plt.imshow(img)
plt.show()

# Primary Model

In [ ]:
!pip install grad-cam

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, CyclicLR
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model Creation with Option for Freezing/Unfreezing
def create_model(num_classes=4, freeze_layers=True):
    model = models.efficientnet_b3(weights='EfficientNet_B3_Weights.DEFAULT')

    if freeze_layers:
        # Freeze all layers initially
        for param in model.parameters():
            param.requires_grad = False
        # Unfreeze only the classifier
        for param in model.classifier.parameters():
            param.requires_grad = True
    else:
        # No freezing: all layers trainable
        for param in model.parameters():
            param.requires_grad = True

    # Simple 4-class classifier
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.classifier[1].in_features, num_classes)
    )

    return model.to(device)

# Accuracy Calculation
def calculate_accuracy(model, data_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Model Training with LR Options and Progressive Layerwise Unfreezing
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="none", progressive_unfreeze=False):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # Learning rate scheduler options
    if lr_scheduler_type == "cyclic":
        scheduler = CyclicLR(optimizer, base_lr=1e-5, max_lr=lr, step_size_up=5, mode="triangular")
    elif lr_scheduler_type == "cosine":
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    else:
        scheduler = None

    # Count total feature blocks (EfficientNet-B3 has 9)
    total_blocks = len(model.features)

    for epoch in range(epochs):
        if progressive_unfreeze:
            blocks_to_unfreeze = min(epoch + 1, total_blocks)
            for param in model.features[-blocks_to_unfreeze:].parameters():
                param.requires_grad = True

            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
            print(f"[Epoch {epoch+1}] Unfrozen blocks: {blocks_to_unfreeze}")
        else:
            blocks_to_unfreeze = "All (no freezing)"

        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            if scheduler and lr_scheduler_type == "cyclic":
                scheduler.step()

            running_loss += loss.item()

        if scheduler and lr_scheduler_type == "cosine":
            scheduler.step()

        val_acc = calculate_accuracy(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Blocks: {blocks_to_unfreeze}, Loss: {running_loss:.4f}, Val Accuracy: {val_acc:.4f}")

    return model

# Model Evaluation
def evaluate_model(model, test_loader):
    test_acc = calculate_accuracy(model, test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    return test_acc

# Grad-CAM Visualization for Random Sample from Each Class (adaptive circular mask)
def generate_gradcam(model, data_loader, target_layer, class_names=["Normal", "Diabetic Retinopathy", "Cataract", "Glaucoma"]):
    model.eval()
    cam = GradCAM(model=model, target_layers=[target_layer])

    all_images, all_labels, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_images.append(images)
            all_labels.append(labels)
            all_preds.append(preds)

    all_images = torch.cat(all_images)
    all_labels = torch.cat(all_labels)
    all_preds = torch.cat(all_preds)

    for class_idx in range(len(class_names)):
        class_indices = (all_labels == class_idx).nonzero(as_tuple=True)[0]
        if len(class_indices) == 0:
            print(f"No samples found for class {class_names[class_idx]}")
            continue

        idx = random.choice(class_indices.tolist())
        image = all_images[idx].unsqueeze(0)
        label = all_labels[idx].item()
        pred = all_preds[idx].item()

        targets = [ClassifierOutputTarget(label)]
        grayscale_cam = cam(input_tensor=image, targets=targets)[0]

        img_np = image.squeeze().cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)

        gray_img = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        gray_blurred = cv2.medianBlur(gray_img, 15)
        circles = cv2.HoughCircles(gray_blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=100, param1=50, param2=30, minRadius=50, maxRadius=0)
        mask = np.zeros(gray_img.shape, dtype=np.uint8)
        if circles is not None:
            circles = np.uint16(np.around(circles))
            for c in circles[0, :1]:
                cv2.circle(mask, (c[0], c[1]), c[2], 1, thickness=-1)
        else:
            h, w, _ = img_np.shape
            center = (w // 2, h // 2)
            radius = min(center) - 5
            cv2.circle(mask, center, radius, 1, thickness=-1)

        grayscale_cam = grayscale_cam * mask
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img_np)
        axes[0].set_title(f"Original: {class_names[label]}")
        axes[0].axis("off")

        pred_status = "Correct" if pred == label else f"Wrong (Pred: {class_names[pred]})"
        axes[1].imshow(cam_image)
        axes[1].set_title(f"Grad-CAM: {pred_status}")
        axes[1].axis("off")
        plt.show()

# DataLoader Creation
def create_dataloaders(train_set, val_set, test_set, batch_size=32):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
# bs = 8, freeze/unfreze = False, epochs = 5, lr = 0.001, scheduler = "none"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=8)
model = create_model(freeze_layers=False)
model = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="none", progressive_unfreeze=False)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])

In [ ]:
# bs = 8, freeze/unfreze = True, epochs = 5, lr = 0.001, scheduler = "none"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=8)
model = create_model(freeze_layers=True)
model = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="none", progressive_unfreeze=True)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])

In [ ]:
# bs = 8, freeze/unfreze = False, epochs = 5, lr = 0.001, scheduler = "cyclic"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=8)
model = create_model(freeze_layers=False)
model = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="cyclic", progressive_unfreeze=False)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])

In [ ]:
# bs = 8, freeze/unfreze = False, epochs = 5, lr = 0.001, scheduler = "cosine"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=8)
model = create_model(freeze_layers=False)
model = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="cosine", progressive_unfreeze=False)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])

In [ ]:
# bs = 12, freeze/unfreze = False, epochs = 5, lr = 0.001, scheduler = "cosine"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=12)
model = create_model(freeze_layers=False)
model = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, lr_scheduler_type="cosine", progressive_unfreeze=False)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])

In [ ]:
# bs = 8, freeze/unfreze = False, epochs = 10, lr = 0.001, scheduler = "cosine"
train_loader, val_loader, test_loader = create_dataloaders(train_set, val_set, test_set, batch_size=8)
model = create_model(freeze_layers=False)
model = train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, lr_scheduler_type="cosine", progressive_unfreeze=False)
evaluate_model(model, test_loader)
generate_gradcam(model, test_loader, target_layer=model.features[-1])